In [ ]:
import yfinance as yf
import pandas as pd
from datetime import datetime, timedelta
import warnings

warnings.filterwarnings('ignore')

# 1. 분석 기간 설정 (최근 3년)
end_date = datetime.today()
start_date = end_date - timedelta(days=365 * 3)

# VOO를 기준으로 나머지 자산 비교 (크립토는 환율 노이즈 제거를 위해 USD 기준 적용)
tickers = ['VOO', 'GOOGL', 'BTC-USD', 'ETH-USD']

print(f"데이터를 불러오는 중... ({start_date.strftime('%Y-%m-%d')} ~ {end_date.strftime('%Y-%m-%d')})")
# yfinance에서 종가 가져오기
df = yf.download(tickers, start=start_date.strftime('%Y-%m-%d'), end=end_date.strftime('%Y-%m-%d'))['Close']

# 2. 데이터 정제 (VOO 영업일 기준 정렬 및 ★일일 수익률 계산★)
df = df.dropna() # 주말/공휴일 등 크립토만 열리는 날 제거 (VOO 기준 동기화)
returns = df.pct_change().dropna()

# 3. 태블로 시각화용: '30일 이동 상관관계 (Rolling Correlation)' 계산
window = 30
rolling_corr = pd.DataFrame()
rolling_corr['GOOGL'] = returns['GOOGL'].rolling(window=window).corr(returns['VOO'])
rolling_corr['BTC'] = returns['BTC-USD'].rolling(window=window).corr(returns['VOO'])
rolling_corr['ETH'] = returns['ETH-USD'].rolling(window=window).corr(returns['VOO'])

# 초기 30일 결측치 제거 및 인덱스(Date)를 컬럼으로 꺼내기
rolling_corr = rolling_corr.dropna().reset_index()

# 타임존 제거 (태블로가 날짜를 에러 없이 인식하도록 깔끔하게 정리)
if rolling_corr['Date'].dt.tz is not None:
    rolling_corr['Date'] = rolling_corr['Date'].dt.tz_localize(None)
rolling_corr['Date'] = rolling_corr['Date'].dt.normalize()

# 4. 최근 상관계수 출력 (리포트 작성용)
print("\n📊 [최근 30일 기준 상관계수 (VOO 대비)]")
print(f"VOO vs GOOGL: {rolling_corr['GOOGL'].iloc[-1]:.3f}")
print(f"VOO vs BTC:   {rolling_corr['BTC'].iloc[-1]:.3f}")
print(f"VOO vs ETH:   {rolling_corr['ETH'].iloc[-1]:.3f}")

# 5. 태블로 친화적 형태 (Long Format - Melt) 변환
# [Date, Asset, Correlation_vs_VOO] 형태로 세로로 길게 변환
tableau_df = rolling_corr.melt(id_vars=['Date'], var_name='Asset', value_name='Correlation_vs_VOO')

# 6. CSV로 저장
csv_filename = "voo_correlation_for_tableau.csv"
tableau_df.to_csv(csv_filename, index=False)
print(f"\n✅ 태블로용 CSV 파일 저장 완료: {csv_filename}")